In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [2]:
img = cv2.imread("siftt1.png")
img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
img = img.astype(np.float32)

In [3]:
# creating scale space
s = 3
num_images = s + 3
k = 2 ** (1/s)
octave_wise_sigmas = {}
s_initial = 1.6
num_octaves = 7

for octave_num in range(1, num_octaves):
    curr_octave_sigmas = []
    for i in range(num_images):
        curr_octave_sigmas.append((2**(octave_num - 1)) * (k**i) * s_initial)
    octave_wise_sigmas[f"octave_{octave_num}"] = curr_octave_sigmas

In [4]:
def get_del_sigma(sigma1, sigma2):
    return np.sqrt(sigma2**2 - sigma1**2)

def gaussian_kernel(size, sigma):
    assert size % 2 == 1, "Kernel size must be odd"
    k = size // 2
    x, y = np.meshgrid(np.arange(-k, k + 1), np.arange(-k, k + 1))
    kernel = np.exp(-(x**2 + y**2) / (2 * sigma**2))
    kernel /= kernel.sum()
    return kernel.astype(np.float32)

def create_dogs_octave(octave_sigmas, img, octave_number):

    assert octave_number > 0

    for a in range(octave_number-1):
        img = cv2.resize(img, (0, 0), fx=.5, fy=.5, interpolation=cv2.INTER_AREA)

    gk_applied_imgs = [img]
    for k in range(1, len(octave_sigmas)):
        del_sigma = get_del_sigma(octave_sigmas[k - 1], octave_sigmas[k])
        kernel_size = int(2*np.ceil(3*del_sigma)+1)
        gk = gaussian_kernel(kernel_size, del_sigma)
        o_gk_img = gk_applied_imgs[-1].copy()

        K = gk.shape[0]
        for i in range(0, gk_applied_imgs[-1].shape[0] - K + 1):
            for j in range(0, gk_applied_imgs[-1].shape[1] - K + 1):
                img_slice = gk_applied_imgs[-1][i:i+K, j:j+K]
                o_gk_img[i + K//2, j + K//2] = np.sum(gk * img_slice)

        gk_applied_imgs.append(o_gk_img)
    dogs = []
    for i in range(1, len(gk_applied_imgs)):
        dogs.append(gk_applied_imgs[i] - gk_applied_imgs[i-1])
    return dogs

def hessian_test(candidate_slice, kernel_dim=3, r=10):

    assert candidate_slice.shape == (3, 3)
    
    slice_center_row = candidate_slice[candidate_slice.shape[0]//2]
    del_square_x = slice_center_row[kernel_dim-1] - 2 * slice_center_row[kernel_dim//2] + slice_center_row[0]

    slice_center_col = candidate_slice[:, kernel_dim//2]
    del_square_y = slice_center_col[kernel_dim-1] - 2 * slice_center_col[kernel_dim//2] + slice_center_col[0]

    del_square_xy = (
    candidate_slice[2, 2]
    - candidate_slice[2, 0]
    - candidate_slice[0, 2]
    + candidate_slice[0, 0]
    ) * 0.25

    det = (del_square_x*del_square_y - del_square_xy**2)
    if det <= 0:
        return False
    if ((del_square_x + del_square_y)**2) / det > (((r + 1)**2) / r):
        return False
    return True

def keypoint_check_condition(slice_3d_img):
    assert slice_3d_img.shape == (3,3,3)
    x = slice_3d_img.flatten()
    center_pxl_index = x.shape[0]//2
    center_pxl_val = x[center_pxl_index] # getting the center pxl value in the 3d slice
    y = np.delete(x, center_pxl_index)
    if np.all(center_pxl_val > y) or np.all(center_pxl_val < y):
        return True
    else:
        return False




In [5]:
dogs_all_octaves = []
for i in range(1, num_octaves):
    dogs_octave = create_dogs_octave(octave_wise_sigmas[f"octave_{i}"], img, i)
    dogs_all_octaves.append(dogs_octave)

keypt_drawn_output_images_all_octaves = []
for z in range(num_octaves-1):
    # skip first and last dog
    kernel_dim = 3
    keypoints_curr_octave = []
    output_imgs_curr_octave = []
    dogs_octave = dogs_all_octaves[z]
    for k in range(1, len(dogs_octave)-1): 
        dogs_3d = np.stack([dogs_octave[k-1], dogs_octave[k], dogs_octave[k+1]])
        output_img = dogs_octave[k].copy()
        
        for i in range(dogs_3d.shape[1]-kernel_dim):
            for j in range(dogs_3d.shape[2]-kernel_dim):
                dogs_3d_slice = dogs_3d[:, i:i+kernel_dim, j:j+kernel_dim]
                if keypoint_check_condition(dogs_3d_slice) and abs(dogs_octave[k][i+1, j+1]) > 8:
                    if hessian_test(dogs_octave[k][i:i+kernel_dim, j:j+kernel_dim]):
                        keypoints_curr_octave.append([i+kernel_dim//2, j+kernel_dim//2])
                        output_img = cv2.circle(output_img, (j+kernel_dim//2, i+kernel_dim//2), 5,  (255, 0, 0), 1)
        output_imgs_curr_octave.append(output_img)

    keypt_drawn_output_images_all_octaves.append(output_imgs_curr_octave)

In [6]:
# x = np.random.randn(100,100 )
# sigma = 3.2
# radius = int(3 * 1.5 * sigma)
# print(radius) # 14
# i, j = x.shape[0]//2, x.shape[1]//2
# a = x[i-radius:i+radius, j-radius: j+radius]

# curr_circle_indices = []
# for p in range(a.shape[0]):
#     for q in range(a.shape[1]):
#         if p**2 + q**2 <= radius **2:
#             # calculate gradient magnitude
#             dx = a[p, q + 1] - a[p, q-1]
#             dy = a[p+1, q] - a[p-1, q]

#             gradient_mag = np.sqrt(dx**2 + dy**2)

#             # calculate gradient direction
#             gradient_angle = np.arctan2(dy, dx)